In [1]:
from __future__ import annotations
import datetime
import os
import sqlite3
from dataclasses import dataclass
from pathlib import Path
from typing import Iterator, List, Optional, Tuple, Dict, Any
import shutil
import time
import math
import glob

import cv2
import numpy as np

# ==== 1. ユーザ設定（ハードコーディング可） =====================================

# 対象ディレクトリ（ここだけ各自の環境に合わせて変更してください）
TARGET_DIR = Path("./data/")  # ★変更推奨

# 出力先を任意パスで指定（TARGET_DIR 配下である必要なし）
dt_now = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
OUTPUT_DIR_PATH = Path(f"./output/{dt_now}")  # ★必要に応じて自由に変更

# TARGET_DIR直下に作るDBファイル名
DB_FILENAME = "faces.sqlite3"

# 顔検出・特徴量抽出
INSIGHTFACE_MODEL_NAME = "antelopev2"          # 例: "antelopev2", "buffalo_l"
DET_SIZE = (640, 640)                          # 推論解像度
MODEL_ROOT = (TARGET_DIR / "_insightface_models")

MIN_DET_CONF = 0.4                              # 低すぎる顔を除外
MIN_FACE_SIZE = 32                              # 幅/高さどちらかがこのピクセル未満を除外

# 動画フレームのサンプリング
FRAMES_PER_VIDEO = 12                           # 各動画から何枚サンプルするか（等間隔）
BUFFER_SEC = 0.5                                # 先頭/末尾のバッファ秒（切り捨て）
MAX_VIDEO_SIDE = 1280                           # フレームの最大辺（速度のため縮小）

# ==== クラスタリング関連パラメータ ===========================================
# ここでは「顔埋め込み（特徴ベクトル）」をクラスタリングして人物IDを分けます。

# PCAで次元圧縮（+whitening）を行うかどうか
# True: 高次元ノイズを抑えてクラスタ分離性を上げやすい（サンプル十分時に有効）
# False: 元の埋め込みをそのまま使用（計算は軽いが、分離が悪化することも）
USE_PCA = True

# PCA後の次元数。サンプル数が十分なときのみ適用されます。
# ↑大きくする: 情報は多く残るがノイズも残りやすい、計算量↑
# ↓小さくする: ノイズ低減・分離性↑になることもあるが潰し過ぎると識別力低下
PCA_DIM = 256

# DBSCANの min_samples（近傍点数の閾値）
# ↑大きくする: 密度要件が厳しくなりノイズ(-1)が増えやすいが、誤結合は減る
# ↓小さくする: クラスタは作りやすいが、スパースな誤クラスタも増えやすい
DBSCAN_MIN_SAMPLES = 3

# 初期eps推定で使う「k近傍」のk
# （各点のk番目に近い距離を分布化→下のパーセンタイルで初期epsを決定）
# ↑大きくする: より遠い近傍まで見るので k距離が大きくなりやすく、初期eps↑（粗く結合）
# ↓小さくする: k距離が小さくなりやすく、初期eps↓（初期から細かく分割）
INITIAL_K_FOR_KNN = 5

# 初期epsを決める際に使うパーセンタイル（%）
# 例: 95.0なら「k距離の95%点」をeps初期値に採用
# ↑大きくする: 初期eps↑（最初は大きく結合→後で分割方向に調整）
# ↓小さくする: 初期eps↓（最初から細かく分割気味）
KNN_PERCENTILE = 95.0

# epsを分割方向に反復調整する倍率（0~1、1に近いほど微調整）
# 反復ごとに eps *= EPS_ADJUST_FACTOR で縮小します
# ↑1に近づける: 調整が緩やか（試行回数が増えるが滑らか）
# ↓小さくする: 収束は速いが過剰分割・不安定になりやすい
EPS_ADJUST_FACTOR = 0.90

# 期待する最低人物数（クラスタ数の下限）。達するまで eps を下げて分割を促進
# ↑大きくする: もっと分割させる（細かく別IDになりやすい、過分割注意）
# ↓小さくする: まとめがち（同一人物に混ざりやすい、過少分割注意）
MIN_EXPECTED_PERSONS = 10

# 「小さすぎるクラスタ」をその他(id_other)へ送る閾値（サンプル数）
# ↑大きくする: 小規模クラスタを弾く→精度保守的、未知/少数データはother行きが増える
# ↓小さくする: 小規模でもIDとして残す→漏れは減るが誤ID増加のリスク
MIN_CLUSTER_SIZE = 5

# その他
DRY_RUN = False                                # Trueならコピーしない（計画のみ）
RNG_SEED = 42
SKIP_OUTPUT_DIR = True                         # 入力探索で出力フォルダ配下を除外
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}
VID_EXTS = {".mp4", ".mov", ".mkv", ".avi", ".m4v", ".webm"}


# ==== 2. 低レベルユーティリティ =====================================================

def is_image(p: Path) -> bool:
    return p.suffix.lower() in IMG_EXTS

def is_video(p: Path) -> bool:
    return p.suffix.lower() in VID_EXTS

def read_image_imdecode(p: Path) -> Optional[np.ndarray]:
    # Unicodeパス対応のためimdecodeを使用
    data = np.fromfile(str(p), dtype=np.uint8)
    if data.size == 0:
        return None
    img = cv2.imdecode(data, cv2.IMREAD_COLOR)
    return img

def write_jpeg(path: Path, img_bgr: np.ndarray, quality: int = 90) -> bool:
    ensure_dir(path.parent)
    ok, buf = cv2.imencode(".jpg", img_bgr, [cv2.IMWRITE_JPEG_QUALITY, int(quality)])
    if not ok:
        return False
    buf.tofile(str(path))
    return True

def l2norm(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    n = np.linalg.norm(x, axis=1, keepdims=True) + eps
    return x / n

def cosine_distances(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    a_n = l2norm(a)
    b_n = l2norm(b)
    return 1.0 - a_n @ b_n.T

def now_iso() -> str:
    return time.strftime("%Y-%m-%dT%H:%M:%S", time.localtime())

def ensure_dir(d: Path):
    d.mkdir(parents=True, exist_ok=True)

def resize_keep_max_side(img: np.ndarray, max_side: int) -> np.ndarray:
    h, w = img.shape[:2]
    m = max(h, w)
    if m <= max_side:
        return img
    scale = max_side / m
    nh, nw = int(round(h * scale)), int(round(w * scale))
    return cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)

def unique_copy(dst_dir: Path, src: Path):
    ensure_dir(dst_dir)
    name = src.name
    stem, ext = src.stem, src.suffix
    out = dst_dir / name
    i = 1
    while out.exists():
        out = dst_dir / f"{stem}__{i}{ext}"
        i += 1
    if not DRY_RUN:
        shutil.copy2(src, out)

def is_subpath(child: Path, parent: Path) -> bool:
    try:
        child.resolve().relative_to(parent.resolve())
        return True
    except Exception:
        return False


# ==== 3. DB（SQLite）レイヤ ===========================================================

class FaceDB:
    """
    files(rel_path UNIQUE, kind, mtime, last_indexed_at)
    faces(file_id, frame_idx, x,y,w,h, conf, emb_dim, embedding BLOB)
    assignments(face_id, person)  # 最新クラスタ結果（毎回クリアして上書き）
    """
    def __init__(self, db_path: Path):
        self.db_path = db_path
        self.conn = sqlite3.connect(str(db_path))
        self.conn.execute("PRAGMA foreign_keys = ON;")
        self._init_schema()

    def _init_schema(self):
        c = self.conn.cursor()
        c.execute("""
        CREATE TABLE IF NOT EXISTS files (
            id INTEGER PRIMARY KEY,
            rel_path TEXT UNIQUE NOT NULL,
            kind TEXT NOT NULL,
            mtime REAL NOT NULL,
            last_indexed_at TEXT NOT NULL
        );""")
        c.execute("""
        CREATE TABLE IF NOT EXISTS faces (
            id INTEGER PRIMARY KEY,
            file_id INTEGER NOT NULL,
            frame_idx INTEGER,
            x INTEGER NOT NULL,
            y INTEGER NOT NULL,
            w INTEGER NOT NULL,
            h INTEGER NOT NULL,
            conf REAL NOT NULL,
            emb_dim INTEGER NOT NULL,
            embedding BLOB NOT NULL,
            FOREIGN KEY(file_id) REFERENCES files(id) ON DELETE CASCADE
        );""")
        c.execute("""
        CREATE TABLE IF NOT EXISTS assignments (
            face_id INTEGER PRIMARY KEY,
            person INTEGER NOT NULL,
            FOREIGN KEY(face_id) REFERENCES faces(id) ON DELETE CASCADE
        );""")
        self.conn.commit()

    def upsert_file(self, rel_path: str, kind: str, mtime: float) -> Tuple[int, bool]:
        """
        returns: (file_id, is_new_or_updated)
        """
        cur = self.conn.cursor()
        cur.execute("SELECT id, mtime FROM files WHERE rel_path = ?;", (rel_path,))
        row = cur.fetchone()
        if row is None:
            cur.execute(
                "INSERT INTO files(rel_path, kind, mtime, last_indexed_at) VALUES (?,?,?,?);",
                (rel_path, kind, mtime, now_iso()),
            )
            self.conn.commit()
            return cur.lastrowid, True
        else:
            file_id, old_mtime = row
            if float(old_mtime) != float(mtime):
                cur.execute(
                    "UPDATE files SET mtime = ?, last_indexed_at = ? WHERE id = ?;",
                    (mtime, now_iso(), file_id),
                )
                # 既存のfacesは削除して取り直す
                cur.execute("DELETE FROM faces WHERE file_id = ?;", (file_id,))
                self.conn.commit()
                return file_id, True
            else:
                return file_id, False

    def insert_face(self, file_id: int, frame_idx: Optional[int], box: Tuple[int,int,int,int],
                    conf: float, emb: np.ndarray):
        x, y, w, h = map(int, box)
        emb = np.asarray(emb, dtype=np.float32).reshape(-1)
        cur = self.conn.cursor()
        cur.execute("""
            INSERT INTO faces(file_id, frame_idx, x,y,w,h, conf, emb_dim, embedding)
            VALUES (?,?,?,?,?,?,?,?,?);
        """, (file_id, frame_idx, x, y, w, h, float(conf), int(emb.size), emb.tobytes()))
        self.conn.commit()

    def get_all_embeddings(self, min_conf: float = 0.0) -> Tuple[np.ndarray, List[Dict[str, Any]]]:
        cur = self.conn.cursor()
        cur.execute("""
            SELECT faces.id, files.id, files.rel_path, faces.frame_idx,
                   faces.x, faces.y, faces.w, faces.h, faces.conf, faces.emb_dim, faces.embedding
            FROM faces
            JOIN files ON faces.file_id = files.id
            WHERE faces.conf >= ?;""", (min_conf,))
        rows = cur.fetchall()
        recs = []
        embs = []
        for (face_id, file_id, rel_path, frame_idx, x,y,w,h, conf, emb_dim, blob) in rows:
            emb = np.frombuffer(blob, dtype=np.float32)
            if emb.size != emb_dim:
                continue
            embs.append(emb)
            recs.append({
                "face_id": face_id,
                "file_id": file_id,
                "rel_path": rel_path,
                "frame_idx": frame_idx,
                "box": (x,y,w,h),
                "conf": conf,
            })
        if len(embs) == 0:
            return np.zeros((0, 0), np.float32), []
        X = np.vstack(embs).astype(np.float32)
        X = l2norm(X)
        return X, recs

    def clear_assignments(self):
        self.conn.execute("DELETE FROM assignments;")
        self.conn.commit()

    def save_assignments(self, mapping_face_to_person: Dict[int, int]):
        cur = self.conn.cursor()
        cur.executemany("INSERT OR REPLACE INTO assignments(face_id, person) VALUES (?,?);",
                        list(mapping_face_to_person.items()))
        self.conn.commit()

    def iter_files(self) -> Iterator[Tuple[int, str, str]]:
        cur = self.conn.cursor()
        for row in cur.execute("SELECT id, rel_path, kind FROM files;"):
            yield row

    def get_face_assignments(self) -> Dict[int, int]:
        cur = self.conn.cursor()
        cur.execute("SELECT face_id, person FROM assignments;")
        return {fid: person for (fid, person) in cur.fetchall()}

    def close(self):
        self.conn.close()


# ==== 4. InsightFace ラッパ ===========================================================

class FaceExtractor:
    """
    InsightFace(app.FaceAnalysis) を使って (bbox, conf, embedding) を抽出。
    - モデルパックは antelopev2 → buffalo_l → buffalo_m → buffalo_s → デフォルト(None) の順にフォールバック
    - CPU固定(ctx_id=-1)。ONNXRuntime-GPU未導入やCUDAなし環境でも安全に動作
    - モデル保存先は TARGET_DIR/_insightface_models
    """
    def __init__(self,
                 model_name: str = INSIGHTFACE_MODEL_NAME,
                 det_size: Tuple[int,int] = DET_SIZE):
        try:
            from insightface.app import FaceAnalysis
        except Exception as e:
            raise RuntimeError(
                "insightface の import に失敗しました。`pip install -U insightface onnxruntime` を実行してください。"
            ) from e

        # モデル保存先を準備
        ensure_dir(MODEL_ROOT)

        # フォールバック候補
        candidates = []
        if model_name is not None:
            candidates.append(model_name)
        candidates += ["buffalo_l", "buffalo_m", "buffalo_s", None]

        last_error = None
        for cand in candidates:
            try:
                # fa = FaceAnalysis(name=cand, root=str(MODEL_ROOT))
                fa = FaceAnalysis(
                        name='antelopev2/antelopev2', 
                        root="./models", 
                        # providers=['CUDAExecutionProvider', 'CPUExecutionProvider'], 
                        allowed_modules=['detection']
                    )
                
                # CPU固定。GPUトラブルやORT差異を避ける
                fa.prepare(ctx_id=-1, det_size=det_size)

                # サニティチェック：detection が実体として載っているか
                models_dict = getattr(fa, "models", {})
                if not isinstance(models_dict, dict) or "detection" not in models_dict:
                    raise RuntimeError(
                        f"モデルパック {cand!r} に detection が見つかりません。"
                    )

                self.app = fa
                self.model_pack = cand if cand is not None else "default"
                print(f"[FaceExtractor] loaded model pack: {self.model_pack}")
                break
            except Exception as e:
                last_error = e
                continue
        else:
            # すべて失敗
            hint = (
                "InsightFace のモデル読込に失敗しました。\n"
                f"- 試行したパック: {candidates}\n"
                f"- 最後のエラー: {last_error}\n\n"
                "対処例:\n"
                "  1) `pip install -U insightface onnxruntime`\n"
                "  2) 既定キャッシュが壊れている場合、以下を削除して再実行\n"
                "     Windows: %USERPROFILE%\\.insightface\\models\\\n"
                "     Linux/Mac: ~/.insightface/models/\n"
                "  3) ファイアウォール/プロキシでモデルDLが遮断されていないか確認\n"
                "  4) 設定で INSIGHTFACE_MODEL_NAME = 'buffalo_l' に変更\n"
            )
            raise RuntimeError(hint)

    def extract_from_image(self, img_bgr: np.ndarray) -> List[Tuple[Tuple[int,int,int,int], float, np.ndarray]]:
        faces = self.app.get(img_bgr)
        recs: List[Tuple[Tuple[int,int,int,int], float, np.ndarray]] = []

        for f in faces:
            # bbox
            x1, y1, x2, y2 = map(float, f.bbox)
            w, h = max(0.0, x2 - x1), max(0.0, y2 - y1)
            if w < MIN_FACE_SIZE or h < MIN_FACE_SIZE:
                continue

            # score
            conf = float(getattr(f, "det_score", 1.0))
            if conf < MIN_DET_CONF:
                continue

            # embedding
            emb = getattr(f, "normed_embedding", None)
            if emb is None or (isinstance(emb, np.ndarray) and emb.size == 0):
                emb = getattr(f, "embedding", None)
            if emb is None:
                continue

            # 1D float32 + L2正規化
            emb = np.asarray(emb, dtype=np.float32).reshape(-1)
            emb = l2norm(emb.reshape(1, -1))[0]

            recs.append(((int(x1), int(y1), int(w), int(h)), conf, emb))

        return recs


# ==== 5. 動画フレームサンプラ =========================================================

def sample_video_frames(path: Path) -> Iterator[Tuple[int, np.ndarray]]:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    if total <= 0:
        total = int(fps * 60)  # 1分仮定
    margin = int(BUFFER_SEC * fps)
    start = max(0, margin)
    end = max(0, total - margin - 1)
    if end <= start:
        end = total - 1
        start = 0
    n = max(1, FRAMES_PER_VIDEO)
    idxs = np.linspace(start, end, num=n, dtype=int)
    seen = set()
    for idx in idxs:
        if idx in seen:
            continue
        seen.add(idx)
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if not ok or frame is None:
            continue
        frame = resize_keep_max_side(frame, MAX_VIDEO_SIDE)
        yield idx, frame
    cap.release()


# ==== 6. インデクシング（DBに未登録だけ抽出） ==========================================

def index_folder(target_dir: Path, db: FaceDB, extractor: FaceExtractor):
    for p in target_dir.rglob("*"):
        # 出力配下はスキップ（OUTPUT_DIR_PATH が TARGET_DIR 内にある場合のみ）
        if SKIP_OUTPUT_DIR and is_subpath(p, OUTPUT_DIR_PATH):
            continue

        if p.is_dir():
            continue
        if not (is_image(p) or is_video(p)):
            continue

        rel = p.relative_to(target_dir).as_posix()
        kind = "image" if is_image(p) else "video"
        mtime = p.stat().st_mtime

        file_id, need_index = db.upsert_file(rel, kind, mtime)
        if not need_index:
            continue  # 既に最新

        print(f"[INDEX] {rel} ({kind})")
        if kind == "image":
            img = read_image_imdecode(p)
            if img is None:
                continue
            recs = extractor.extract_from_image(img)
            for (box, conf, emb) in recs:
                db.insert_face(file_id, None, box, conf, emb)
        else:
            for frame_idx, frame in sample_video_frames(p):
                recs = extractor.extract_from_image(frame)
                for (box, conf, emb) in recs:
                    db.insert_face(file_id, int(frame_idx), box, conf, emb)


# ==== 7. クラスタリング（PCA→DBSCAN、eps自動調整） ====================================

def cluster_faces(db: FaceDB) -> Dict[int, int]:
    """
    returns: face_id -> person_id（0,1,2,...）; ノイズや小クラスタは -1
    """
    X, recs = db.get_all_embeddings(min_conf=MIN_DET_CONF)
    if X.shape[0] == 0:
        print("[CLUSTER] embeddings=0")
        return {}

    # オプション: PCA + whitening + 再正規化
    X_proc = X.copy()
    if USE_PCA and X.shape[0] >= max(PCA_DIM + 32, 64) and X.shape[1] > PCA_DIM:
        from sklearn.decomposition import PCA
        pca = PCA(n_components=PCA_DIM, whiten=True, random_state=RNG_SEED)
        X_proc = pca.fit_transform(X_proc).astype(np.float32)
        X_proc = l2norm(X_proc)

    # 初期eps: k-NN距離の上位パーセンタイル
    from sklearn.neighbors import NearestNeighbors
    nn = NearestNeighbors(n_neighbors=min(INITIAL_K_FOR_KNN, len(X_proc)), metric="cosine")
    nn.fit(X_proc)
    dists, _ = nn.kneighbors(X_proc)
    kth = dists[:, -1]  # 各点のk番目近傍
    eps = float(np.percentile(kth, KNN_PERCENTILE))

    def run_dbscan(eps_val: float):
        from sklearn.cluster import DBSCAN
        dbs = DBSCAN(eps=eps_val, min_samples=DBSCAN_MIN_SAMPLES, metric="cosine", n_jobs=-1)
        labels = dbs.fit_predict(X_proc)
        return labels

    labels = run_dbscan(eps)
    def n_clusters_valid(lbl: np.ndarray) -> int:
        return int(len(set(lbl)) - (1 if -1 in lbl else 0))

    # 「最低でもこの人数」まで分割方向（eps↓）に調整
    tries = 0
    while n_clusters_valid(labels) < MIN_EXPECTED_PERSONS and eps > 1e-4 and tries < 40:
        eps *= EPS_ADJUST_FACTOR
        labels = run_dbscan(eps)
        tries += 1

    # 小クラスタを -1 扱いへ
    from collections import Counter
    cnt = Counter([l for l in labels if l != -1])
    small = {lab for lab, c in cnt.items() if c < MIN_CLUSTER_SIZE}
    labels = np.array([(-1 if (l == -1 or l in small) else l) for l in labels], dtype=int)

    # 人物IDをサイズ順で 0.. に再番号付け（安定な並び）
    sizes = Counter([l for l in labels if l != -1])
    order = [lab for (lab, _) in sizes.most_common()]
    remap = {lab: i for i, lab in enumerate(order)}
    mapped = np.array([remap.get(l, -1) for l in labels], dtype=int)

    # face_id -> person
    face_ids = [r["face_id"] for r in recs]
    face_to_person = {fid: int(pid) for fid, pid in zip(face_ids, mapped)}
    print(f"[CLUSTER] faces={len(face_ids)} persons={n_clusters_valid(mapped)} eps_final={eps:.4f}")
    return face_to_person


# ==== 8. 出力コピー（ファイル単位の仕分け） ============================================

def copy_by_person(target_dir: Path, db: FaceDB, mapping: Dict[int, int]):
    # face_id -> person を file 単位に集約
    cur = db.conn.cursor()
    q = """
    SELECT faces.id, faces.file_id, files.rel_path
    FROM faces JOIN files ON faces.file_id = files.id;
    """
    file_to_persons: Dict[int, set] = {}
    for face_id, file_id, rel_path in cur.execute(q):
        pid = mapping.get(face_id, -1)
        file_to_persons.setdefault(file_id, set()).add(pid)

    output_root = OUTPUT_DIR_PATH
    ensure_dir(output_root)

    # id_* / id_other にコピー
    for file_id, rel_path, kind in db.iter_files():
        src = target_dir / rel_path
        persons = file_to_persons.get(file_id, set())
        if not persons:
            persons = {-1}
        for pid in persons:
            sub = "id_other" if pid == -1 else f"id_{pid:03d}"
            dst_dir = output_root / sub
            print(f"[COPY] {rel_path} -> {dst_dir.name}/")
            unique_copy(dst_dir, src)


# ==== 9. 代表顔サムネ生成（各人物フォルダに1枚だけ） ================================

def _clamp_box(x: int, y: int, w: int, h: int, W: int, H: int) -> Tuple[int,int,int,int]:
    x = max(0, min(x, W-1))
    y = max(0, min(y, H-1))
    w = max(1, min(w, W - x))
    h = max(1, min(h, H - y))
    return x, y, w, h

def _read_video_frame_resized(path: Path, frame_idx: int) -> Optional[np.ndarray]:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return None
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ok, frame = cap.read()
    cap.release()
    if not ok or frame is None:
        return None
    return resize_keep_max_side(frame, MAX_VIDEO_SIDE)

def save_representative_crops(target_dir: Path, db: FaceDB):
    """
    assignments から各 person の代表 face を選び、id_xxx/ に _____base_id_{face_id}_____.jpg を1枚保存。
    - person == -1 (id_other) はスキップ
    - 代表選択は conf の最大値（同点なら面積大）
    - 動画はクラスタ時と同じ縮小後スケールで切り出すため、フレームを同じように縮小してからcrop
    """
    cur = db.conn.cursor()
    rows = cur.execute("""
        SELECT a.person, f.id, f.conf, f.x, f.y, f.w, f.h, f.frame_idx, files.rel_path, files.kind
        FROM assignments AS a
        JOIN faces AS f ON a.face_id = f.id
        JOIN files ON f.file_id = files.id
        WHERE a.person >= 0;
    """).fetchall()

    # person ごとに最良faceを選定
    best: Dict[int, Tuple] = {}
    for person, face_id, conf, x, y, w, h, frame_idx, rel_path, kind in rows:
        area = int(w) * int(h)
        key = (float(conf), area)
        if person not in best or key > best[person][0]:
            best[person] = (key, (person, face_id, conf, x, y, w, h, frame_idx, rel_path, kind))

    for person, (_, rec) in best.items():
        _, face_id, conf, x, y, w, h, frame_idx, rel_path, kind = rec
        src_path = target_dir / rel_path

        # 画像読み込み
        if kind == "image":
            img = read_image_imdecode(src_path)
        else:
            img = _read_video_frame_resized(src_path, int(frame_idx) if frame_idx is not None else 0)

        if img is None:
            print(f"[BASE] skip (cannot read): {rel_path}")
            continue

        H, W = img.shape[:2]
        x, y, w, h = _clamp_box(int(x), int(y), int(w), int(h), W, H)
        crop = img[y:y+h, x:x+w].copy()
        if crop.size == 0:
            print(f"[BASE] skip (empty crop): {rel_path}")
            continue

        # 保存先（既存の _____base_id_*_____.jpg は削除して常に1枚に保つ）
        person_dir = OUTPUT_DIR_PATH / f"id_{person:03d}"
        ensure_dir(person_dir)
        for old in person_dir.glob("_____base_id_*_____.jpg"):
            try:
                old.unlink()
            except Exception:
                pass

        out_path = person_dir / f"_____base_id_{face_id}_____.jpg"
        ok = True if DRY_RUN else write_jpeg(out_path, crop)
        if ok:
            print(f"[BASE] saved: {out_path}")
        else:
            print(f"[BASE] failed: {out_path}")


# ==== 10. メイン =====================================================================

def main():
    np.random.seed(RNG_SEED)

    if not TARGET_DIR.exists():
        raise FileNotFoundError(f"TARGET_DIR not found: {TARGET_DIR}")

    # 出力ディレクトリを作成（入力探索ではスキップする可能性あり）
    ensure_dir(OUTPUT_DIR_PATH)

    # DB
    db_path = TARGET_DIR / DB_FILENAME
    db = FaceDB(db_path)

    # 顔抽出器
    extractor = FaceExtractor()

    # (1) 未登録/更新のみインデクシング
    index_folder(TARGET_DIR, db, extractor)

    # (2) DB → 埋め込み取得 & クラスタ
    face_to_person = cluster_faces(db)
    db.clear_assignments()
    db.save_assignments(face_to_person)

    # (3) ファイルを人物ごとにコピー
    copy_by_person(TARGET_DIR, db, face_to_person)

    # (4) 各人物フォルダに代表顔1枚を保存
    save_representative_crops(TARGET_DIR, db)

    db.close()
    print("[DONE]")

if __name__ == "__main__":
    main()


c:\Users\hiahara\Documents\code\python_util\__実装系__\face_classify\.venv\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:121: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: ./models\models\antelopev2/antelopev2\1k3d68.onnx landmark_3d_68
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: ./models\models\antelopev2/antelopev2\2d106det.onnx landmark_2d_106
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: ./models\models\antelopev2/antelopev2\genderage.onnx genderage
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: ./models\models\antelopev2/antelopev2\glintr100.onnx recognition
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: ./models\models\antelopev2/antelopev2\scrfd_10g_bnkps.onnx detection [1, 3, '?', '?'] 127.5 128.0
set det-size: (640, 640)
[FaceExtractor] loaded model pack: antelopev2
[INDEX] 日向坂46/けやき坂46 ロゴ.png (image)
[INDEX] 日向坂46/スクリーンショット 20